In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("API Key is set") 

API Key is set


In [2]:
from langchain_openai import ChatOpenAI


In [3]:
llm = ChatOpenAI(model="gpt-5-nano", temperature = 0)

In [4]:
response =llm.invoke("What is AI? Tell me in 1 line")

response.content

'AI is the field of building computer systems that can perform tasks that normally require human intelligence—like learning, reasoning, planning, perception, and language understanding.'

## **RAG IMPLEMENTATION WITH PDF**


#### **Extracting Text from Pdf**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ("./Docs/fabric-onelake.pdf")

loader = PyPDFLoader(pdf_path)

docs = loader.load()

docs


[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26056.01', 'creator': 'Microsoft Learn', 'creationdate': '2026-04-04T11:36:57+00:00', 'title': 'fabric onelake | Microsoft Learn', 'moddate': '2026-04-04T11:36:57+00:00', 'source': './Docs/fabric-onelake.pdf', 'total_pages': 431, 'page': 0, 'page_label': '1'}, page_content="Tell us about your PDF experience.\nMicrosoft OneLake documentation\nMicrosoft OneLake is Fabric's single, unified, logical data lake for the whole organization.\nOneLake comes automatically with every Fabric tenant with no infrastructure to manage.\nAbout OneLake\nｅOVERVIEW\nWhat is OneLake?\nOneLake security\nOneLake catalog\nOneLake access and APIs\n｀DEPLOY\nImplement medallion lakehouse architecture\nｂGET STARTED\nCreate a lakehouse with OneLake\nOneLake file explorer\nFind data in the OneLake catalog\nUse Iceberg tables in OneLake\nOneLake shortcuts\nｐCONCEPT\nWhat are shortcuts?\nｂGET STARTED\nCreate a shortcut\nｃHOW-TO GUIDE"),
 Document(metadata={'prod

#### Creating own Metadata for pdf chunks

In [6]:
for i in docs:
    i.metadata = {"source": "fabric-onelake.pdf",
                  "developer": "Microsoft"}

In [7]:
docs[0].metadata

{'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}

#### **STEP 2: SPLITTING THE DOCUMENT INTO CHUNKS**

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 100
)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content="Tell us about your PDF experience.\nMicrosoft OneLake documentation\nMicrosoft OneLake is Fabric's single, unified, logical data lake for the whole organization.\nOneLake comes automatically with every Fabric tenant with no infrastructure to manage.\nAbout OneLake\nｅOVERVIEW\nWhat is OneLake?\nOneLake security\nOneLake catalog\nOneLake access and APIs\n｀DEPLOY\nImplement medallion lakehouse architecture\nｂGET STARTED\nCreate a lakehouse with OneLake\nOneLake file explorer\nFind data in the OneLake catalog\nUse Iceberg tables in OneLake\nOneLake shortcuts\nｐCONCEPT\nWhat are shortcuts?\nｂGET STARTED\nCreate a shortcut\nｃHOW-TO GUIDE"),
 Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content='Access shortcuts\nOneLake and Azure integration\nｃHOW-TO GUIDE\nIntegrate OneLake with Azure Databricks\nIntegrate OneLake with Azure HDInsight\nIntegrate OneLake with Azu

In [9]:
len(chunks)

469

### **STEP 3: CREATING EMBEDDINGS FOR THE CHUNKS**

In [10]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [11]:
#embedding_model.embed_query("WHAT IS AI?")

### **STEP 4" CREATE AND STORE EMBEDDINGS IN VECTOR STORE**

In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents= chunks,
    embedding= embedding_model
)

In [13]:
#vectors = []
#for doc in chunks:
#    emb = embedding_model.embed_documents(doc.page_content)
#    vectors.append(emb)

#vectors

#### **STEP 5: SEMANTIC SEARCH**

In [14]:
vectorstore.similarity_search("What do we need onelake?", k = 3)

[Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content="OneLake uses zone-redundant storage (ZRS) where available to protect against datacenter\nfailures. You can also enable business continuity and disaster recovery (BCDR) for a capacity to\nreplicate your data to a secondary geographic region.\nFor more information, see Plan for disaster recovery and data protection.\nReady to start using OneLake? Here's how to get started:\nOneLake file explorer for Windows\nOneLake shortcuts\nRecover deleted files in OneLake\nOneLake access and APIs\nLast updated on 02/12/2026\nNext steps\nCreate your first lakehouse with OneLake\nRelated content"),
 Document(metadata={'developer': 'Microsoft', 'source': 'fabric-onelake.pdf'}, page_content="Since OneLake supports the same APIs as ADLS and Blob Storage, many open source\nlibraries and packages compatible with ADLS and Blob Storage work seamlessly with\nOneLake (for example, Azure Storage Explorer). Other librarie

#### **TALK TO LLM**

In [15]:
context = vectorstore.similarity_search("What is AI?", k = 3)


response = llm.invoke(f"What is AI? You can answer using following context: {context}")

print(response.content)

AI stands for Artificial Intelligence—computers or software that can perform tasks that usually require human intelligence, such as understanding language, learning, and making inferences.

In the context of the document you provided, AI is used to power transformations on unstructured text files (.txt) through Microsoft Fabric’s AI-powered shortcut transformations. These transformations apply language processing to extract insights and produce a queryable Delta table. Capabilities include:
- Summarization: generate concise summaries from long-form text
- Translation: translate text between supported languages
- Sentiment analysis: label text sentiment as positive, negative, or neutral
- PII detection: find and redact personally identifiable information (names, phone numbers, emails)
- Name recognition: extract named entities like people, organizations, locations

Key notes:
- Output is kept in sync with the source files automatically (Delta table updated as the source changes).
- AI t